In [16]:
import numpy as np

# ===============================================================
#  FIXED PARAMETERS
# ===============================================================
Fs_in = 9e6      # Input sampling rate
L     = 2        # Interpolation factor
M     = 3        # Decimation factor
N     = 32768    # Number of input samples

# FORMATS
Q_data = 15      # Input data is s16.15
Q_coef = 19      # Coefficients are 20-bit (s1.19)
Coef_Width = 20  # Bit width of coefficients

stim_file = "frac_stimulus.txt"
exp_file  = "frac_expected.txt"

coef_hex = [
"ffff2","ffebd","ffc44","ff8e9","ff676","ff6fc","ffb2d","0012c","00554","004d1","0002f",
"ffb63","ffac1","fff22","004bb","00636","001ac","ffaed","ff872","ffd42","00575","00936",
"00426","ffa3c","ff4e1","ffa18","005f7","00d4e","00816","ffa03","ff042","ff546","005cc",
"01275","00de8","ffaaa","fea84","fee45","00485","018d9","01650","ffcbb","fe364","fe42c",
"00179","020e1","0228d","0010b","fda32","fd51f","ffb77","02ba6","03576","00968","fcd27",
"fbc94","fefa6","03c42","056e9","01ac1","fb661","f8b77","fd44b","05f2d","0a853","04c0b",
"f758d","ee199","f5d25","119fd","36415","50167","50167","36415","119fd","f5d25","ee199",
"f758d","04c0b","0a853","05f2d","fd44b","f8b77","fb661","01ac1","056e9","03c42","fefa6",
"fbc94","fcd27","00968","03576","02ba6","ffb77","fd51f","fda32","0010b","0228d","020e1",
"00179","fe42c","fe364","ffcbb","01650","018d9","00485","fee45","fea84","ffaaa","00de8",
"01275","005cc","ff546","ff042","ffa03","00816","00d4e","005f7","ffa18","ff4e1","ffa3c",
"00426","00936","00575","ffd42","ff872","ffaed","001ac","00636","004bb","fff22","ffac1",
"ffb63","0002f","004d1","00554","0012c","ffb2d","ff6fc","ff676","ff8e9","ffc44","ffebd",
"ffff2"
]

# ===============================================================
#  PARSING LOGIC FOR 20-BIT SIGNED HEX
# ===============================================================
coef_int = []
max_val_20b = 1 << (Coef_Width - 1) # 2^19

for h in coef_hex: #two’s complement.
    val = int(h, 16)
    if val >= max_val_20b:  # sign extend
        val -= (1 << Coef_Width)
    coef_int.append(val)

coef_int = np.array(coef_int)
coef_float = coef_int / (2**Q_coef)  # normalize to s1.19

print(f"Loaded {len(coef_int)} coefficients.")

# ===============================================================
#  STIMULUS GENERATION (wideband)
# ===============================================================
t = np.arange(N) / Fs_in
x = (0.45*np.sin(2*np.pi*0.2e6*t) )
x /= np.max(np.abs(x)) * 1.1
x_int = np.round(x * (2**Q_data)).astype(np.int16)

with open(stim_file, "w") as f:
    for v in x_int: f.write(f"{int(v)}\n")
print(f"Generated {stim_file} with {N} samples")

x_float = x_int / (2**Q_data)
x_up = np.zeros(len(x_float)*L); x_up[::L] = x_float
y_fir = np.convolve(x_up, coef_float, mode='full')[:len(x_up)]
y_decim = y_fir[::M]
y_int = np.clip(np.round(y_decim * (2**Q_data)), -32768, 32767).astype(np.int16)

with open(exp_file, "w") as f:
    for v in y_int: f.write(f"{int(v)}\n")

Fs_out = Fs_in * L / M
print(f"Generated {exp_file} with {len(y_int)} samples")
print("\n=== FRACTIONAL DECIMATOR PERFORMANCE SUMMARY ===")
print(f"Input Fs         = {Fs_in/1e6:.3f} MHz")
print(f"Output Fs        = {Fs_out/1e6:.3f} MHz")
print(f"Conversion Ratio = {L}/{M} = {L/M:.4f}")
print("Bandwidth reduction factor:", Fs_out/Fs_in)

# ===============================================================
#  FREQUENCY RESPONSE CALCULATION
# ===============================================================
N_imp = max(4096, len(coef_int))
imp = np.zeros(N_imp); imp[0] = 1.0
imp_up = np.zeros(len(imp)*L); imp_up[::L] = imp
H_time = np.convolve(imp_up, coef_float, mode='full')[:len(imp_up)]

N_fft_h = 1 << 18
H_fir = np.fft.rfft(H_time, N_fft_h)
f = np.fft.rfftfreq(N_fft_h, 1/(Fs_in * L))

H_mag = np.abs(H_fir)
H_db = 20*np.log10(np.maximum(H_mag / np.max(H_mag), 1e-15))

F_PASS = 2.8e6
F_STOP = 3.2e6
pb_mask = f <= F_PASS
tb_mask = (f >= F_PASS) & (f <= F_STOP)
sb_mask = f >= F_STOP

pb_vals = H_db[pb_mask]
pb_ripple_span = np.max(pb_vals) - np.min(pb_vals)
pb_ripple_pk = pb_ripple_span / 2.0

if np.any(tb_mask):
    tb_start_db_idx = np.where(f >= F_PASS)[0][0]
    tb_end_db_idx = np.where(f <= F_STOP)[0][-1]
    tb_start_db = H_db[tb_start_db_idx]
    tb_end_db   = H_db[tb_end_db_idx]
else:
    tb_start_db = float('nan')
    tb_end_db   = float('nan')

if np.any(sb_mask):
    sb_vals = H_db[sb_mask]
    sb_peak_mag = np.max(sb_vals)
    sb_attn_db = -sb_peak_mag
else:
    sb_attn_db = float('nan')
# ===============================================================
#  REQUIREMENTS CHECK SUMMARY
# ===============================================================
print("\n=== REQUIREMENTS CHECK (UPDATED COEFFS) ===")
print(f"Passband ripple (±)      : {pb_ripple_pk:.3f} dB  (span {pb_ripple_span:.3f} dB)  [spec ≤ ±0.25 dB]")
print(f"Transition band drop     : {tb_start_db:.2f} → {tb_end_db:.2f} dB  [spec 2.8–3.2 MHz (tight)]")
print(f"Stopband attenuation     : {sb_attn_db:.2f} dB  [spec ≥ 80 dB beyond 3.2 MHz]")


print("\nDONE.")


Loaded 144 coefficients.
Generated frac_stimulus.txt with 32768 samples
Generated frac_expected.txt with 21846 samples

=== FRACTIONAL DECIMATOR PERFORMANCE SUMMARY ===
Input Fs         = 9.000 MHz
Output Fs        = 6.000 MHz
Conversion Ratio = 2/3 = 0.6667
Bandwidth reduction factor: 0.6666666666666666

=== REQUIREMENTS CHECK (UPDATED COEFFS) ===
Passband ripple (±)      : 0.124 dB  (span 0.247 dB)  [spec ≤ ±0.25 dB]
Transition band drop     : -0.25 → -82.84 dB  [spec 2.8–3.2 MHz (tight)]
Stopband attenuation     : 82.33 dB  [spec ≥ 80 dB beyond 3.2 MHz]

DONE.
